In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_ITO, Delhi - CPCB.xlsx",skiprows=16)

In [4]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene
0,01-01-2025 00:00,02-01-2025 00:00,155.56,215.50,19.51,58.88,47.17,34.65,8.69,1.56,15.39,2.55
1,02-01-2025 00:00,03-01-2025 00:00,180.56,238.41,20.22,58.39,47.48,40.68,9.55,1.91,14.70,2.51
2,03-01-2025 00:00,04-01-2025 00:00,245.67,337.46,28.66,122.75,88.59,68.09,16.95,3.54,14.35,2.52
3,04-01-2025 00:00,05-01-2025 00:00,212.56,266.54,27.34,115.91,83.88,66.86,12.80,2.96,12.73,2.56
4,05-01-2025 00:00,06-01-2025 00:00,112.02,148.31,20.97,63.51,50.82,43.35,14.70,0.95,12.19,2.58
...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,385.88,512.17,10.27,58.35,38.95,15.59,16.10,2.17,27.63,2.39
316,13-11-2025 00:00,14-11-2025 00:00,333.64,471.96,10.65,58.87,39.94,16.69,10.71,1.93,25.84,2.38
317,14-11-2025 00:00,15-11-2025 00:00,262.57,385.21,9.66,45.84,32.23,14.85,17.04,1.70,26.54,2.40
318,15-11-2025 00:00,16-11-2025 00:00,277.16,400.71,10.29,58.07,39.25,15.76,19.59,1.99,26.69,2.41


In [7]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   From Date  320 non-null    object 
 1   To Date    320 non-null    object 
 2   PM2.5      311 non-null    float64
 3   PM10       311 non-null    float64
 4   NO         319 non-null    float64
 5   NO2        319 non-null    float64
 6   NOx        319 non-null    float64
 7   NH3        319 non-null    float64
 8   SO2        319 non-null    float64
 9   CO         319 non-null    float64
 10  Ozone      319 non-null    float64
 11  Benzene    319 non-null    float64
dtypes: float64(10), object(2)
memory usage: 30.1+ KB


In [6]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 12)


In [8]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 1
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
dtype: int64


In [9]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [10]:




# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (319, 12)
          From Date           To Date   PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  155.56  215.50  19.51  58.88  47.17   
1  02-01-2025 00:00  03-01-2025 00:00  180.56  238.41  20.22  58.39  47.48   
2  03-01-2025 00:00  04-01-2025 00:00   63.75  122.72  28.66  65.61  51.07   
3  04-01-2025 00:00  05-01-2025 00:00   63.75  266.54  27.34  65.61  83.88   
4  05-01-2025 00:00  06-01-2025 00:00  112.02  148.31  20.97  63.51  50.82   

     NH3    SO2    CO  Ozone  Benzene  
0  34.65   8.69  1.56  15.39     2.55  
1  40.68   9.55  1.91  14.70     2.51  
2  27.28  16.95  1.20  14.35     2.52  
3  27.28  12.80  1.20  12.73     2.56  
4  43.35  14.70  0.95  12.19     2.58  


In [11]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [12]:
df.to_excel('ITODelhi2025.xlsx', index=False)